# Chapitre 16 - Cours 1 : boosting et chemins de régularisation

Ce premier notebook couvre la section 16.2 du livre. Nous y montrons que le
**boosting avec rétrécissement** (« shrinkage ») se comporte comme un **lasso** posé
dans l'espace gigantesque de toutes les fonctions de base possibles (par exemple
tous les arbres), puis nous en tirons trois leçons :

1. la régression *forward stagewise* incrémentale (algorithme 16.1) suit
   approximativement le chemin du lasso (figure 16.1) ;
2. le principe du **« pari sur la parcimonie »** (*bet on sparsity*) explique
   pourquoi une pénalité $L_1$ est préférable en grande dimension ;
3. les chemins de régularisation du boosting sont « bien élevés » (monotones,
   figures 16.3–16.4) et convergent vers une solution de **marge $L_1$ maximale**
   (figure 16.5).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl.ch16_ensembles import (standardize, forward_stagewise, lasso_path_cd,
                                ridge_path_df, block_correlated_data,
                                l1_arc_length, adaboost_shrunk)

## 1. Qu'est-ce qu'un ensemble ?

L'**apprentissage ensembliste** construit un prédicteur en combinant de nombreux
modèles simples, appelés *apprenants de base* ou *apprenants faibles*. Nous en avons
déjà rencontré plusieurs :

- le **bagging** (section 8.7) et les **forêts aléatoires** (chapitre 15) : un
  comité d'arbres vote ;
- le **boosting** (chapitre 10) : le comité est construit séquentiellement et les
  votes sont pondérés ;
- le **stacking** (section 8.8) ;
- plus généralement toute méthode à *dictionnaire* (splines de régression, etc.),
  où les fonctions de base jouent le rôle des apprenants faibles.

On peut décomposer la démarche en deux tâches : (i) produire une population
d'apprenants de base à partir des données ; (ii) les combiner en un prédicteur final.

> **Remarque (codes correcteurs d'erreurs).** Un ancêtre des ensembles est la méthode
> ECOC pour la classification à $K$ classes : chaque classe $k$ reçoit un mot de code
> binaire $(C_{k1},\dots,C_{kL})$ (une ligne de la matrice de codage, tableau 16.1 du
> livre) ; on apprend un classifieur à deux classes par colonne $\ell$, qui estime
> $\hat p_\ell(x)$, la probabilité que le bit $\ell$ vaille 1 ; puis on affecte $x$ à la
> classe minimisant $\delta_k(x)=\sum_{\ell=1}^L |C_{k\ell}-\hat p_\ell(x)|$. Les bits
> redondants permettent de « corriger » quelques erreurs des classifieurs ; en pratique
> le gain vient surtout d'une réduction de variance, comme pour le bagging.

## 2. Régression pénalisée sur un dictionnaire de fonctions de base

Notons $\mathcal T=\{T_k\}_{k=1}^K$ le dictionnaire de **tous** les arbres de régression à
$J$ feuilles que l'on peut construire sur les données d'apprentissage ($K$ est
astronomique). On considère le modèle linéaire (éq. (16.1) du livre)
$$
f(x)=\sum_{k=1}^K \alpha_k\,T_k(x).
$$
Comme $K\gg N$, les moindres carrés seuls n'ont aucun sens (une infinité de solutions
interpolent les données) : il faut **régulariser**. On résout donc (éq. (16.2))
$$
\hat\alpha(\lambda)=\arg\min_{\alpha}\ \sum_{i=1}^N\Big(y_i-\sum_{k=1}^K\alpha_kT_k(x_i)\Big)^2+\lambda\,J(\alpha),
$$
avec $J(\alpha)=\sum_k\alpha_k^2$ (ridge, éq. (16.3)) ou $J(\alpha)=\sum_k|\alpha_k|$
(lasso, éq. (16.4)).

> **Rappel (lasso, section 3.4).** Pour $\lambda$ modéré ou grand, la solution du lasso
> est **parcimonieuse** : beaucoup de $\hat\alpha_k(\lambda)$ sont exactement nuls. Les
> coefficients non nuls sont de plus *rétrécis* : $|\hat\alpha_k(\lambda)|<|\hat\alpha_k(0)|$.
> Lorsque $\lambda$ croît, tous les coefficients diminuent et finissent par s'annuler.

L'idée est séduisante : seule une petite fraction des arbres est sans doute utile pour
approcher une fonction cible donnée. Mais $K$ est bien trop grand pour résoudre (16.2)
directement. C'est ici qu'intervient une stratégie *pas à pas*.

## 3. L'algorithme forward stagewise incrémental (algorithme 16.1)

1. Initialiser $\check\alpha_k=0$ pour tout $k$ ; choisir un petit pas $\varepsilon>0$ et un
   grand nombre d'itérations $M$.
2. Pour $m=1,\dots,M$ :
   - (a) $(\beta^*,k^*)=\arg\min_{\beta,k}\sum_i\big(y_i-\sum_l\check\alpha_lT_l(x_i)-\beta T_k(x_i)\big)^2$ ;
   - (b) $\check\alpha_{k^*}\leftarrow\check\alpha_{k^*}+\varepsilon\cdot\operatorname{signe}(\beta^*)$.
3. Renvoyer $f_M(x)=\sum_k\check\alpha_kT_k(x)$.

Au départ tous les coefficients sont nuls (ce qui correspond à $\lambda=\infty$) ; à
chaque étape un seul coefficient bouge, d'une quantité minuscule.

**Comment calculer l'étape (a).** Notons $r=y-\sum_l\check\alpha_l\mathbf T_l$ le résidu
courant et $\mathbf t_k=(T_k(x_1),\dots,T_k(x_N))^T$. Pour $k$ fixé, la fonction
$\beta\mapsto\|r-\beta\mathbf t_k\|^2$ est un trinôme du second degré en $\beta$ ; sa
dérivée $-2\,\mathbf t_k^T(r-\beta\mathbf t_k)$ s'annule en
$\beta_k=\mathbf t_k^Tr/\|\mathbf t_k\|^2$. En développant le carré, le minimum vaut
$$
\|r-\beta_k\mathbf t_k\|^2=\|r\|^2-2\beta_k\,\mathbf t_k^Tr+\beta_k^2\|\mathbf t_k\|^2
=\|r\|^2-\frac{(\mathbf t_k^Tr)^2}{\|\mathbf t_k\|^2}.
$$
Minimiser en $k$ revient donc à **maximiser** $(\mathbf t_k^Tr)^2/\|\mathbf t_k\|^2$ ; si
les bases sont standardisées, c'est choisir la base la plus **corrélée** (en valeur
absolue) au résidu, et $\operatorname{signe}(\beta^*)=\operatorname{signe}(\mathbf t_{k^*}^Tr)$.
C'est ce que code `forward_stagewise` du module `esl/ch16_ensembles.py`.